# Lecture 15 · Assignment 1. Can you trust a gradient?

In Lecture 15 we checked a handful of backprop derivatives against a numerical slope and found a real bug
(`=` instead of `+=`). This assignment turns that one-off check into a proper tool, and then uses it to hunt bugs,
to find the places where it gives wrong answers, and to understand gradient accumulation better than "remember to zero the grads".

**What you will do**
1. Gradient-check **every** parameter of the network and report a relative error
2. Hunt five planted bugs and work out which ones a gradient check can and cannot catch
3. Plant a ReLU kink on purpose and watch a correct network fail its gradient check
4. Predict what repeated `backward()` calls do, fix `backward()` so it behaves like PyTorch, and use accumulation on purpose
5. Use PyTorch as a second, independent referee for every one of the 41 gradients

Cells marked **Exercise** contain `raise NotImplementedError` for you to replace. Each one is followed by a
**check** cell that prints ✅, ❌ or ⏳. Cells marked **Your answer** want a few sentences of explanation. Those are
graded too, and they matter as much as the code. Run the notebook from top to bottom. A later exercise often uses
the function you wrote in an earlier one. Part 5 needs `torch`, which you already imported in the lecture notebook.

In [112]:
import math, random

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

## Setup: the code from Lecture 15

These are the `Scalar`, `Neuron`, `Layer` and `MLP` classes from the lecture, together with the reorder dataset,
`compute_loss` and `train`. Do not edit them. Several exercises swap a method out temporarily and expect the original to be here.

In [125]:
class Scalar:
    def __init__(self, value, label='', oprt='', children=()):
        self.value = value
        self.label = label
        self.oprt = oprt
        self.children = children
        self.der = 0.0
        self._backward = lambda: None

    def __add__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value + other.value, '', oprt='+', children=(self, other))
        def _backward():
            self.der += 1.0 * out.der
            other.der += 1.0 * out.der
        out._backward = _backward
        return out

    def __radd__(self, other):          # other + self
        return self + other

    def __rmul__(self, other):          # other * self
        return self * other

    def __pow__(self, n):
        out = Scalar(self.value ** n, label='', children=(self,), oprt=f'**{n}')
        def _backward():
            self.der += n * (self.value ** (n-1)) * out.der
        out._backward = _backward
        return out

    def __neg__(self):
        return self * -1

    def __sub__(self, other):
        return self + (-other)

    def __rsub__(self, other):          # other - self
        return other + (-self)

    def __mul__(self, other):
        other = other if isinstance(other, Scalar) else Scalar(other)
        out = Scalar(self.value * other.value, '', oprt='*', children=(self, other))
        def _backward():
            self.der += other.value * out.der
            other.der += self.value * out.der
        out._backward = _backward
        return out

    def relu(self):
        out = Scalar(0 if self.value < 0 else self.value, '', oprt='ReLU', children=(self,))
        def _backward():
            self.der += (out.value > 0) * out.der
        out._backward = _backward
        return out

    def backward(self):
        visited = set()
        topo = list()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v.children:
                    build_topo(child)
                topo.append(v)
        build_topo(self)
        self.der = 1.0
        for v in reversed(topo):
            v._backward()

In [126]:
class Neuron:
    def __init__(self, n_inputs, act='relu', initial_bias=None):
        self.weights = [Scalar(random.uniform(-1,1), label=f'w{i+1}') for i in range(n_inputs)]
        self.bias = Scalar(random.uniform(-1,1), label='b') if initial_bias == None else Scalar(initial_bias)
        self.act = act
        self.last_pre = None              # pre-activation of the latest call, for debugging

    def __call__(self, x):
        n = sum(xi * wi for xi, wi in zip(x, self.weights)) + self.bias
        self.last_pre = n.value
        if self.act == 'relu':
            return n.relu()
        return n                          # linear

    @property
    def params(self):
        return self.weights + [self.bias]


class Layer:
    def __init__(self, n_inputs, n_outputs, **kwargs):
        self.neurons = [Neuron(n_inputs, **kwargs) for _ in range(n_outputs)]

    def __call__(self, x):
        outputs = [n(x) for n in self.neurons]
        return outputs[0] if len(outputs) == 1 else outputs

    @property
    def params(self):
        return [p for neuron in self.neurons for p in neuron.params]


class MLP:
    def __init__(self, n_inputs, output_layers, act='relu', initial_bias=None):
        sizes = [n_inputs] + output_layers
        self.layers = [Layer(sizes[i], sizes[i+1], act=act, initial_bias=initial_bias)
                       for i in range(len(output_layers))]

    def __call__(self, x):
        for l in self.layers:
            x = l(x)
        return x

    @property
    def params(self):
        return [p for l in self.layers for p in l.params]

In [127]:
# x = [demand, stock, incoming],   y (order) = max(0, demand - stock - incoming)
xs = [
    [3.0, 1.0, 1.0],   # 3 - 1 - 1 =  1
    [4.0, 1.0, 0.0],   # 4 - 1 - 0 =  3
    [2.0, 2.0, 1.0],   # 2 - 2 - 1 = -1  -> 0
    [5.0, 2.0, 1.0],   # 5 - 2 - 1 =  2
    [1.0, 0.0, 2.0],   # 1 - 0 - 2 = -1  -> 0
]
ys = [1.0, 3.0, 0.0, 2.0, 0.0]

def compute_loss(nn):
    ypred = [nn(x) for x in xs]
    return sum((ya - yp)**2 for ya, yp in zip(ys, ypred))

def train(model, lr=0.01, epochs=500, zero_grad=True):
    loss_history = []
    for step in range(epochs):
        loss = compute_loss(model)                       # 1. forward pass
        if zero_grad:
            for p in model.params: p.der = 0.0           # 2. zero grad
        loss.backward()                                  # 3. backward pass
        for p in model.params: p.value += -p.der * lr    # 4. update
        loss_history.append(loss.value)
    return loss_history

## Part 1. Check every parameter

The lecture's `grad_check` printed eight hand-picked parameters, and we eyeballed the two columns. That does not scale.
We want a function that checks **all** parameters and gives a single number we can test.

Comparing raw differences is not enough. A difference of `0.001` is negligible when the gradient is `500`, but alarming when the
gradient is `0.002`. So we use a **relative error**. The `max(1.0, ...)` in the denominator stops it from blowing up when both
gradients are close to zero.

### Exercise 1.1. `grad_check_all`

Write `grad_check_all(model, loss_fn=compute_loss, h=1e-6)`. It returns a list with one tuple
`(index, backprop, numerical, rel_err)` for **every** parameter in `model.params`, in order. Requirements:

* Zero every parameter's `der` before calling `backward()`. The check below deliberately leaves garbage in `der` first.
* Use the central difference `(L(p+h) - L(p-h)) / (2h)` for the numerical derivative.
* Restore every parameter to its original value afterwards. The model must be unchanged when you return.

In [128]:
def rel_error(a, b):
    return abs(a - b) / max(1.0, abs(a), abs(b))

In [129]:
def grad_check_all(model, loss_fn=compute_loss, h=1e-6):
    # compute loss and back-prop its gradients
    loss = loss_fn(model)
    for p in model.params:
            p.der = 0.0
    
    loss.backward() # back-propogation
    result = []

    for i in range(len(model.params)):
        p = model.params[i]
        old = p.value # saving, the old-value!

        p.value = old + h
        loss_plus = loss_fn(model).value
        p.value = old - h
        loss_minus = loss_fn(model).value

        numerical_loss = (loss_plus - loss_minus) / (2 * h)
        rel_loss = rel_error(p.der, numerical_loss)
        result.append((i, p.der, numerical_loss, rel_loss))

        # restoring param's value
        p.value = old
    
    return result
    

In [130]:
def test_grad_check_all():
    ok = True
    for seed in range(5):
        random.seed(seed)
        model = MLP(3, [4, 4, 1])
        before = [p.value for p in model.params]
        for p in model.params:
            p.der = 123.0                              # stale garbage: your function must clear it
            # NOTE: the above one should be done by me?
        report = grad_check_all(model)
        worst = max(r[3] for r in report)
        print(f"   seed {seed}: {len(report)} params checked, worst relative error {worst:.1e}")
        ok &= len(report) == 41 and worst < 1e-6
        ok &= [p.value for p in model.params] == before
        ok &= [r[0] for r in report] == list(range(41))
    return ok

check("grad_check_all", test_grad_check_all)

   seed 0: 41 params checked, worst relative error 1.5e-09
   seed 1: 41 params checked, worst relative error 1.1e-09
   seed 2: 41 params checked, worst relative error 1.1e-09
   seed 3: 41 params checked, worst relative error 7.8e-10
   seed 4: 41 params checked, worst relative error 1.1e-09
✅ grad_check_all


Every error should be around `1e-9`, which is roughly as close as two different ways of computing a float can agree.
That is what "the gradients are right" looks like.

## Part 2. Bug hunt

Below are five broken versions of pieces of `Scalar`. The helper `installed(name)` swaps one of them in for the duration of a
`with` block and puts the original back afterwards.

| bug | what changed |
|---|---|
| `A` | `__mul__` assigns gradients with `=` instead of `+=` |
| `B` | `__pow__` uses `n * value**n` as its local derivative (wrong exponent) |
| `C` | `relu` passes the gradient through even when the output was clipped to 0 |
| `D` | `__rsub__` computes `self - other` instead of `other - self` |
| `E` | `backward()` skips the topological sort and visits nodes depth-first from the output, running each `_backward` as soon as it reaches the node |

Read each one carefully before you answer anything.

In [88]:
from contextlib import contextmanager

def mul_bug_A(self, other):
    other = other if isinstance(other, Scalar) else Scalar(other)
    out = Scalar(self.value * other.value, '', oprt='*', children=(self, other))
    def _backward():
        self.der = other.value * out.der               # BUG: = instead of +=
        other.der = self.value * out.der
    out._backward = _backward
    return out

def pow_bug_B(self, n):
    out = Scalar(self.value ** n, '', oprt=f'**{n}', children=(self,))
    def _backward():
        self.der += n * (self.value ** n) * out.der    # BUG: should be n-1
    out._backward = _backward
    return out

def relu_bug_C(self):
    out = Scalar(0 if self.value < 0 else self.value, '', oprt='ReLU', children=(self,))
    def _backward():
        self.der += out.der                            # BUG: forgot the (out.value > 0) mask
    out._backward = _backward
    return out

def rsub_bug_D(self, other):
    return self + (-other)                             # BUG: computes self - other

def backward_bug_E(self):
    visited = set()
    self.der = 1.0
    def visit(v):                                      # BUG: no topological sort
        if v in visited:
            return
        visited.add(v)
        v._backward()
        for child in v.children:
            visit(child)
    visit(self)

BUGS = {'A': ('__mul__', mul_bug_A), 'B': ('__pow__', pow_bug_B), 'C': ('relu', relu_bug_C),
        'D': ('__rsub__', rsub_bug_D), 'E': ('backward', backward_bug_E)}

@contextmanager
def installed(name):
    attr, broken = BUGS[name]
    original = Scalar.__dict__[attr]
    setattr(Scalar, attr, broken)
    try:
        yield
    finally:
        setattr(Scalar, attr, original)

### Exercise 2.1. Predict before you run

For each bug, imagine we build `random.seed(2); MLP(3, [4, 4, 1])` with the bug installed and then

* **caught**: run `grad_check_all` on it. Does at least one parameter have a relative error above `1e-5`?
* **trains**: run `train(model)` for the usual 500 steps at `lr=0.01`. Is the final loss below `0.05`? (The correct code reaches about `0.0097`.)

Fill in `True` or `False` for all ten entries **without running anything first**. Then run the check, and for every
prediction you got wrong, write down why in the answer cell that follows.

*Hints.* Which of these bugs change the forward pass, and which change only the backward pass? Which parameters are used more than
once in the loss? Does the loss in `compute_loss` contain `float - Scalar`?

In [89]:
predictions = {
    #      caught  trains
    'A': (True,   True),
    # 'B': (None,   None),
    # 'C': (None,   None),
    # 'D': (None,   None),
    # 'E': (None,   None),
}

In [90]:
def run_bug(name):
    with installed(name):
        random.seed(2)
        model = MLP(3, [4, 4, 1])
        caught = max(r[3] for r in grad_check_all(model)) > 1e-5
        random.seed(2)
        model = MLP(3, [4, 4, 1])
        final = train(model)[-1]
    return caught, final < 0.05, final

def check_predictions():
    if any(v is None for pair in predictions.values() for v in pair):
        raise NotImplementedError
    ok = True
    for name, (p_caught, p_trains) in predictions.items():
        caught, trains, final = run_bug(name)
        mark = lambda guess, actual: '✓' if guess == actual else '✗'
        print(f"   bug {name}: caught {caught!s:5} (you said {p_caught!s:5} {mark(p_caught, caught)})   "
              f"trains {trains!s:5} (you said {p_trains!s:5} {mark(p_trains, trains)})   final loss {final:.4g}")
        ok &= (p_caught == caught) and (p_trains == trains)
    return ok

check("bug predictions", check_predictions)

   bug A: caught True  (you said True  ✓)   trains False (you said True  ✗)   final loss 10
❌ bug predictions


**Your answer.** For every bug, explain in one or two sentences why it is or is not caught, and why training does or does not still work.

_(write here)_

### Exercise 2.2. The smallest witness

A good bug report comes with the smallest example that shows the bug. For each of bugs `A` and `E`, write a function of
three `Scalar` inputs `v = [a, b, c]` that uses **as few operations as possible** and fails a gradient check when the bug is
installed. It must still pass with the correct code. An *operation* is any node with a non-empty `oprt`. Remember that
`x - y` and `-x` are built from several operations.

* `witness_A` must use exactly **1** operation.
* `witness_E` may use at most **3** operations. Think about which kind of graph the depth-first order gets wrong.

`gradcheck_fn(f, values)` below runs a numerical check on a plain function. You may use it while you experiment.

In [131]:
def numerical_grad_fn(f, values, h=1e-6):
    grads = []
    for i in range(len(values)):
        plus  = [Scalar(v + h if j == i else v) for j, v in enumerate(values)]
        minus = [Scalar(v - h if j == i else v) for j, v in enumerate(values)]
        grads.append((f(plus).value - f(minus).value) / (2 * h))
    return grads

def gradcheck_fn(f, values, tol=1e-5):
    vs = [Scalar(v) for v in values]
    f(vs).backward()
    return all(rel_error(v.der, n) < tol for v, n in zip(vs, numerical_grad_fn(f, values)))

def count_ops(root):
    seen, stack = set(), [root]
    while stack:
        v = stack.pop()
        if v not in seen:
            seen.add(v)
            stack.extend(v.children)
    return sum(1 for v in seen if v.oprt)

In [132]:
def witness_A(v):
    a, b, c = v
    # YOUR CODE HERE
    raise NotImplementedError

def witness_E(v):
    a, b, c = v
    # YOUR CODE HERE
    raise NotImplementedError

In [133]:
def test_witness(name, fn, max_ops):
    values = [1.5, -2.0, 0.7]
    n_ops = count_ops(fn([Scalar(v) for v in values]))
    good = gradcheck_fn(fn, values)
    with installed(name):
        bad = not gradcheck_fn(fn, values)
    print(f"   witness_{name}: {n_ops} ops, passes when correct: {good}, fails with bug {name}: {bad}")
    return good and bad and n_ops <= max_ops

check("witness_A", lambda: test_witness('A', witness_A, 1))
check("witness_E", lambda: test_witness('E', witness_E, 3))

⏳ witness_A: not implemented yet
⏳ witness_E: not implemented yet


**Your answer.** `d + d*c` fails under bug E but `d*c + d` passes. Explain why the order of the two terms matters, and what that tells you about testing for this kind of bug.

_(write here)_

### Exercise 2.3. A test that *can* see bug D

A gradient check only compares backward against forward. If the forward pass is wrong, both sides are wrong in the same way.
Write `forward_test()` that checks the **values** produced by every operator `Scalar` supports against plain Python
arithmetic: `+`, `-`, `*`, `**`, unary `-`, `relu`, and the reflected forms `2 + s`, `2 - s`, `2 * s`.
It returns `True` if everything matches and `False` otherwise.

In [ ]:
def forward_test():
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def test_forward_test():
    results = {}
    results['correct'] = forward_test()
    for name in 'ABCDE':
        with installed(name):
            results[name] = forward_test()
    print("   ", results)
    return results['correct'] and not results['D']

check("forward_test passes on correct code and catches bug D", test_forward_test)

**Your answer.** Bug D was invisible on our loss. Give a loss function, or a single line of a training loop, where bug D would silently make training go wrong. Explain what would go wrong.

_(write here)_

## Part 3. When a correct network fails its gradient check

ReLU has a corner at 0. If a pre-activation is within `h` of zero, then `L(p+h)` and `L(p-h)` sit on **different sides** of
the corner, and the central difference averages two different slopes. Backprop picks one side. A correct network then fails.

### Exercise 3.1. Plant a kink

Write `plant_kink(model, layer, neuron, sample, margin)`. It changes **only the bias** of
`model.layers[layer].neurons[neuron]` so that this neuron's pre-activation on input `xs[sample]` becomes exactly `margin`.
*Hint: the bias enters the pre-activation with coefficient 1. Run the model on `xs[sample]` and read `last_pre`.*
Careful with deeper layers. Their inputs depend on earlier layers, but you only change the bias of the chosen neuron,
so its inputs do not move.

In [139]:
def plant_kink(model, layer, neuron, sample, margin):
    model(xs[sample])

    layers = model.layers[layer]
    neuron = layers.neurons[neuron]

    shift_needed = margin - neuron.last_pre
    neuron.bias.value = neuron.bias.value + shift_needed
    

In [140]:
def test_plant():
    ok = True
    for layer, neuron, sample in [(0, 2, 0), (1, 0, 3), (2, 0, 1)]:
        random.seed(2)
        model = MLP(3, [4, 4, 1])
        weights_before = [w.value for l in model.layers for n in l.neurons for w in n.weights]
        plant_kink(model, layer, neuron, sample, 1e-4)
        model(xs[sample])
        pre = model.layers[layer].neurons[neuron].last_pre
        ok &= abs(pre - 1e-4) < 1e-12
        ok &= weights_before == [w.value for l in model.layers for n in l.neurons for w in n.weights]
    return ok

check("plant_kink", test_plant)

✅ plant_kink


In [145]:
random.seed(2)
model = MLP(3, [4, 4, 1])
plant_kink(model, layer=0, neuron=2, sample=0, margin=1e-4)
for h in (1e-2, 1e-3, 1e-6):
# for h in (1e-2,):
    bad = [(i, round(b, 4), round(n, 4)) for i, b, n, e in grad_check_all(model, h=h) if e > 1e-5]
    print(f"h = {h:.0e}: {len(bad)} mismatches", bad[:5])

h = 1e-02: 7 mismatches [(8, -1.0842, -0.9151), (9, -0.6957, -0.5312), (10, -0.3908, -0.3156), (11, -0.3092, -0.234), (32, 0.0, 0.0772)]
h = 1e-03: 4 mismatches [(8, -1.0842, -1.0613), (9, -0.6957, -0.6772), (10, -0.3908, -0.3889), (11, -0.3092, -0.3073)]
h = 1e-06: 0 mismatches []


**Your answer.** Explain the table above. Which parameters fail and why exactly those? The margin is `1e-4`. Why does `h = 1e-3` still fail for all four parameters of that neuron, and what is the condition on `h` for a weight `w_i`? What does this mean for the habit of gradient-checking a trained network?

_(write here)_
1. Which parameters fail, and why those?

▎ Parameters 8–11 fail because they're the only ones that appear in this neuron's pre-activation, which was planted at 1e-4, just above ReLU's corner. A large enough nudge pushes pre below 0 in L(p-h), so the central difference averages slope 0 and slope 1, while backprop uses only slope 1. Other parameters can't move this pre, so their gradients still match.

2. Why does h = 1e-3 fail for all four parameters?

▎ The sample is [3, 1, 1], so pre = 3·w1 + w2 + w3 + b. At h = 1e-3, nudging w1 moves pre by 3e-3, and nudging w2, w3 or b moves it by 1e-3. All of these are larger than the margin of 1e-4, so all four cross the corner.

3. What is the condition on h for a weight w_i?

▎ Nudging w_i by h moves pre by |x_i|·h, where x_i is that weight's input. The nudge crosses the corner when |x_i|·h > margin, which means h > margin / |x_i|.
▎
▎ For w1, this gives h > 1e-4 / 3 ≈ 3.3e-5. For w2 and w3, it gives h > 1e-4. That's why w1 was the only parameter to fail at h = 1e-4. The bias has a coefficient of 1, so it follows the same rule with |x_i| = 1.

Crossing is required for a failure, but the error must also exceed the 1e-5 threshold to be reported. In a deeper layer, x_i is the output of the previous layer, not the raw input.

4. What does this mean for gradient-checking a trained network?

▎ Training can move pre-activations close to the corner. Two of our targets are exactly 0, so the network pushes those outputs towards 0, which puts their pre-activations near ReLU's corner. A trained network can therefore fail a gradient check even when its code is correct.
▎
▎ A failure should be treated as a reason to investigate, not as proof of a bug. Check whether the failing parameters feed a neuron with pre close to 0, and re-run with a smaller h. If the mismatch disappears, it was a kink. If it stays at a very small h, suspect a real bug.

## Part 4. Accumulation: the bug and the feature

### Exercise 4.1. Predict

In the lecture, forgetting `zero_grad` made gradients pile up across training steps. There, every step built a **new** graph,
so only the parameters (leaves) were shared between steps. What happens if we call `backward()` several times on the
**same** graph?

For `x = Scalar(1.0)` and `L = ((x * 2) * 3) * 4`, write down `x.der` after 1, 2 and 3 calls to `L.backward()`, with
no zeroing in between. Work it out on paper by tracking `der` of every intermediate node. Do not guess `24, 48, 72`.

In [150]:
predicted_x_der = [24, 96, 240]   # after 1, 2, 3 calls

In [152]:
def test_predicted():
    if None in predicted_x_der:
        raise NotImplementedError
    x = Scalar(1.0)
    L = ((x * 2) * 3) * 4
    actual = []
    for _ in range(3):
        L.backward()
        actual.append(x.der)
    print(f"   you said {predicted_x_der}, actual {actual}")
    return actual == predicted_x_der

check("repeated backward on one graph", test_predicted)

   you said [24, 96, 240], actual [24.0, 96.0, 240.0]
✅ repeated backward on one graph


### Exercise 4.2. Make `backward()` behave like PyTorch

PyTorch keeps gradients only on leaves. Calling `backward()` twice (with `retain_graph=True`) simply adds the same gradient again,
so you get `24, 48, 72`. Our `Scalar` also stores `der` on every intermediate node, and those stale values compound.

Write `backward_leaf_accumulate(self)`. It works like `backward()`, but before propagating it resets `der` to `0.0` on
every **non-leaf** node of the graph (a node with children). Leaves keep accumulating as before.

In [175]:
def backward_leaf_accumulate(self):
    visited = set()
    topo = list()
    def build_topo(v):
        if v not in visited:
            visited.add(v)
            if len(v.children) > 0:
                for child in v.children:
                    build_topo(child)
                v.der = 0.0
            topo.append(v)
    build_topo(self)
    self.der = 1.0
    for v in reversed(topo):
        v._backward()


In [176]:
def test_leaf_backward():
    x = Scalar(1.0)
    L = ((x * 2) * 3) * 4
    ders = []
    for _ in range(3):
        backward_leaf_accumulate(L)
        ders.append(x.der)
    random.seed(2)
    model = MLP(3, [4, 4, 1])
    loss = compute_loss(model)
    backward_leaf_accumulate(loss)
    once = [p.der for p in model.params]
    backward_leaf_accumulate(loss)
    twice = [p.der for p in model.params]
    print("   x.der after 1, 2, 3 calls:", ders)
    return ders == [24.0, 48.0, 72.0] and all(abs(t - 2 * o) < 1e-9 for o, t in zip(once, twice))

check("backward_leaf_accumulate", test_leaf_backward)

   x.der after 1, 2, 3 calls: [24.0, 48.0, 72.0]
✅ backward_leaf_accumulate


### Exercise 4.3. Accumulation on purpose: micro-batches

Our loss is a sum over training rows, and the derivative of a sum is the sum of the derivatives. So we never need the graph
for the whole batch at once. We can build the graph for **one row**, call `backward()`, throw the graph away, move to the next
row, and let the parameters' `der` add up. Large language models are trained this way when a batch does not fit in GPU
memory. It is called *gradient accumulation*.

Write `accumulated_grads(model)`. Zero the parameters once. Then for each training row build only that row's loss
`(y - model(x))**2` and call `backward()` on it. Return the list of `p.der` for all parameters.

In [177]:
def accumulated_grads(model):
    n = len(xs)
    for i in range(n):
        y = model(xs[i])
        loss = (ys[i] - y) ** 2
        loss.backward()

    return [p.der for p in model.params]

In [178]:
def test_accumulated():
    ok = True
    for seed in range(5):
        random.seed(seed)
        model = MLP(3, [4, 4, 1])
        acc = accumulated_grads(model)
        loss = compute_loss(model)
        for p in model.params:
            p.der = 0.0
        loss.backward()
        full = [p.der for p in model.params]
        ok &= max(abs(a - f) for a, f in zip(acc, full)) < 1e-12
    return ok

check("micro-batch gradients equal full-batch gradients", test_accumulated)

✅ micro-batch gradients equal full-batch gradients


**Your answer.** (a) Why is it safe here to call `backward()` once per row without zeroing in between, when Exercise 4.1 showed that repeated `backward()` calls compound? (b) If the loss were the **mean** over rows instead of the sum, what would you have to change so that the accumulated gradient still matches?

TODO: revisit it twice intuition for me isn't 100% apt
(a) over here intermediate nodes aren't shared hence derivates value is zero
(b) the gradient would still match > say mean is  (1/N)(loss_1 + loss_2 + loss_3 + .... + loss_N) > I would have called loss function as (loss_1/N).backward() and so on.

## Part 5. A second referee: PyTorch

The numerical check is slow and, as Part 3 showed, sometimes misleading. An independent implementation of the *same*
mathematics is a stronger referee. PyTorch knows nothing about our code, so if both agree to 12 digits we can be confident.

### Exercise 5.1. Copy our network into PyTorch

Write `to_torch(model)` that returns a `torch.nn.Sequential` computing **exactly** what our `MLP` computes, with the
same weights:

* one `torch.nn.Linear(n_in, n_out)` per layer, followed by `torch.nn.ReLU()`. Our lecture `MLP` applies ReLU on the output layer too.
* row `j` of a `Linear`'s `weight` holds neuron `j`'s weights, and `bias[j]` holds its bias.
* use `float64` everywhere (`torch.set_default_dtype` is already set below), or the comparison will only agree to about 7 digits.

Then write `torch_grads(tmodel)` that returns the gradients as a flat list **in the same order as `model.params`**
(neuron by neuron, weights then bias).

In [186]:
import torch
torch.set_default_dtype(torch.float64)
X_t = torch.tensor(xs)
Y_t = torch.tensor(ys).unsqueeze(1)          # shape (5, 1)

In [187]:
def to_torch(model):
    modules = []
    for layer in model.layers:
        n_in  = len(layer.neurons[0].weights)   # inputs per neuron
        n_out = len(layer.neurons)              # neurons in this layer
        lin = torch.nn.Linear(n_in, n_out)

        with torch.no_grad():
            lin.weight.copy_(torch.tensor([[w.value for w in n.weights] for n in layer.neurons]))
            lin.bias.copy_(torch.tensor([n.bias.value for n in layer.neurons]))

        modules += [lin, torch.nn.ReLU()]       # ReLU after every layer, output layer too
    return torch.nn.Sequential(*modules)

def torch_grads(tmodel):
    grads = []
    for m in tmodel:
        if isinstance(m, torch.nn.Linear):      # skip the ReLU modules: they have no parameters
            for j in range(m.out_features):     # neuron by neuron...
                grads += m.weight.grad[j].tolist()
                grads.append(m.bias.grad[j].item()) 
    return grads

In [188]:
def test_torch():
    ok = True
    for seed in list(range(10)) + ['0 trained']:
        random.seed(0 if seed == '0 trained' else seed)
        model = MLP(3, [4, 4, 1])
        if seed == '0 trained':
            train(model)                                   # 500 steps, as in the lecture
        tmodel = to_torch(model)
        t_loss = ((tmodel(X_t) - Y_t) ** 2).sum()
        t_loss.backward()
        loss = compute_loss(model)
        for p in model.params:
            p.der = 0.0
        loss.backward()
        diff = max(abs(a - p.der) for a, p in zip(torch_grads(tmodel), model.params))
        n_zero = sum(p.der == 0 for p in model.params)
        print(f"   seed {seed!s:9}: loss ours {loss.value:9.5f} torch {t_loss.item():9.5f}   "
              f"max grad diff {diff:.1e}   zero grads {n_zero:2d}/41")
        ok &= abs(loss.value - t_loss.item()) < 1e-10 and diff < 1e-10
    return ok

check("our gradients match PyTorch", test_torch)

   seed 0        : loss ours  10.00719 torch  10.00719   max grad diff 1.8e-15   zero grads  8/41
   seed 1        : loss ours   7.20867 torch   7.20867   max grad diff 4.4e-16   zero grads 25/41
   seed 2        : loss ours   3.10834 torch   3.10834   max grad diff 4.4e-15   zero grads 14/41
   seed 3        : loss ours   9.75674 torch   9.75674   max grad diff 3.6e-15   zero grads 21/41
   seed 4        : loss ours  11.21186 torch  11.21186   max grad diff 1.8e-15   zero grads 18/41
   seed 5        : loss ours  80.32242 torch  80.32242   max grad diff 5.7e-14   zero grads 19/41
   seed 6        : loss ours   6.78047 torch   6.78047   max grad diff 3.6e-15   zero grads 14/41
   seed 7        : loss ours  12.32980 torch  12.32980   max grad diff 1.8e-15   zero grads 31/41
   seed 8        : loss ours  12.52106 torch  12.52106   max grad diff 1.8e-15   zero grads 20/41
   seed 9        : loss ours  13.17566 torch  13.17566   max grad diff 0.0e+00   zero grads 33/41
   seed 0 trained: l

**Your answer.** Look at the "zero grads" column. (a) Even at initialization, between 8 and 33 of the 41 gradients are exactly 0. Why? (b) After training, seed 0 has 41 of 41 gradients exactly 0, and both libraries agree. Is that a bug? What does it tell you about this model, and what will further training do?

NOTE:Fill it